# Web Scraping and Data Cleaning of Tour de France Rider Histor

## Introduction

This notebook covers the data cleaning and export phase of the Tour de France Data Science Project.

Our primary data source is [Wikipedia’s list of Tour de France general classification winners](https://en.wikipedia.org/wiki/List_of_Tour_de_France_general_classification_winners), which provides comprehensive information on every champion since 1903.

We will use web scraping to extract the raw HTML tables, clean and restructure the critical data, and export everything into reproducible formats (**CSV**, **JSON**, and **Pickle**). These cleaned datasets lay the groundwork for all subsequent analysis and visualization.


## Focus of This Notebook

- Ensuring the data is consistent, readable, and analysis-ready  
- Making every step transparent and well-documented for easy re-use by others

## Libraries and Tools Used

The following Python libraries will be used and should be installed before running the notebook:

- **requests**: HTTP requests to fetch web pages  
- **BeautifulSoup**: Parse HTML and extract the needed content  
- **csv**: Export cleaned data as CSV  
- **json**: Export structured data as JSON  
- **re**: Use regular expressions for precise cleaning  
- **pickle**: Save datasets for quick reload in future notebooks


In [29]:
# !pip install requests
# !pip install beautifulsoup4

In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import json
import re
import pickle

## `tdf_winner_nationalities_clean.csv`

### Dataset

We use the cleaned dataset: `tdf_winner_nationalities_clean.csv`.

This dataset provides a tidy summary of the nationalities of all Tour de France general classification winners.  
Using the “By nationality” table scraped from Wikipedia, we’ve organized, standardized, and exported:

- **Country**  
- **Number of unique Tour de France winners for that nation**  
- **List of those winning cyclists** (for reference or deeper study)

This compact dataset allows us to quickly answer:  
**Which country has produced the most unique Tour de France winners?**

It serves as a reproducible foundation for historical comparison and visual exploration.


In [2]:
URL = "https://en.wikipedia.org/wiki/List_of_Tour_de_France_general_classification_winners#By_nationality"

# Set headers to imitate a regular web browser
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Fetch the main Wikipedia page
page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, "html.parser")


# Find the main content division
div = soup.find("div", {"class": "mw-content-ltr mw-parser-output"})

# Locate the table listing winners by nationality (usually 4th wikitable)
table_nationality = div.find_all("table", {"class": "wikitable"})[3]


# Extract all data rows (skip the header row)
rows = table_nationality.find_all("tr")[1:]

data = []
for row in rows:
    country = row.find("th")   # Extract country cell (from 'th')
    wins = row.find_all("td")[0]  # Extract number of wins (first 'td')
    winning_cyclist = row.find_all("td")[1]  # Extract winning cyclists (second 'td')
    # Clean and append to results
    data.append([
        country.text.strip(),
        wins.text.strip(),
        winning_cyclist.text.strip()
    ])

# Write results to a CSV file
header = ["Country", "Winners", "Winning_Cyclists"]
with open('data/tdf_winner_nationalities_clean.csv','w',encoding='utf-8',newline='') as f:
    writer = csv.writer(f) # f is the file object and we pass it to the csv.writer
    writer.writerow(header) # write the header (first row)
    writer.writerows(data) # write all data rows (data is a list of lists)

## `tdf_multiple_winners_clean.json`

This dataset contains clean, structured information on cyclists who have won the Tour de France more than three times, as obtained by scraping the “Multiple winners of the Tour de France general classification” table from Wikipedia – List of Tour de France general classification winners.

Each record provides:

- The cyclist’s name  
- The cyclist’s surname  
- The total number of Tour de France wins by that cyclist  
- The specific years in which each win was achieved  
- The cyclist’s nationality, if available
- The link to the year of the win on Wikipedia

The purpose of building and saving this file is to enable a fast and reproducible answer to:  
**“Which cyclists have won the Tour more than three times, and in which years?”**

All cleaned data is stored as a structured JSON file:

`data/tdf_multiple_winners_clean.json`

This format is optimal for semi-structured records like name, win count, and list of years, and will be easy to reload for future analyses or sharing.

In [32]:
URL = "https://en.wikipedia.org/wiki/List_of_Tour_de_France_general_classification_winners#By_nationality"

# Headers to mimic a real browser visit 
headers = {
    "User-Agent": "Mozilla/5.0"
}

page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, "html.parser")

div = soup.find("div",{"class":"mw-content-ltr mw-parser-output"})

table_multiple_winners = div.find_all("table", {"class":"wikitable"})[2]
rows = table_multiple_winners.find_all("tr")[1:]  # Skip header row

cyclist_data = []

for row in rows:
    cols = row.find_all("td")

    cyclist_surname = re.search(r'-value="(.+), ',str(row))
    cyclist_name = re.search(r', (\w+)"',str(row))
    country = re.search(r'<abbr title="(.+)"',str(row))

    wins_td = cols[-1]
    len_wins = len(wins_td.find_all("a"))
    years = [a.text for a in wins_td.find_all("a")]
    links_years = [f"https://en.wikipedia.org{a['href']}" for a in wins_td.find_all("a")]

    wins_years = [{"year": y, "link": l} for y, l in zip(years, links_years)]
    
    cyclist_data.append({
        "surname": cyclist_surname.group(1),
        "name": cyclist_name.group(1),
        "country": country.group(1) if country else None,
        "win_count": len_wins,
        "wins_years": wins_years
    })

# Export to JSON
with open('data/tdf_multiple_winners_clean.json', 'w', encoding='utf-8') as f:
    json.dump(cyclist_data, f, indent=3)


## `tdf_shortest_tour_clean.p`

This dataset provides clean, structured data on the length of every Tour de France in both days and kilometers, as extracted from the “Tour de France general classification winners” table on Wikipedia – List of Tour de France general classification winners.

Each record in this dataset contains:

- The year of the edition  
- The country of the winner  
- The full name of the winner  
- The total distance of that Tour in kilometers  
- The winner’s total time (“race days”) as displayed (if available; sometimes this is “hours:minutes”, other times needs to be computed from text!)

In [33]:
URL = "https://en.wikipedia.org/wiki/List_of_Tour_de_France_general_classification_winners#By_nationality"

# Headers to mimic a real browser visit 
headers = {
    "User-Agent": "Mozilla/5.0"
}

page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, "html.parser")

div = soup.find("div",{"class":"mw-content-ltr mw-parser-output"})

table_classification_winners = div.find_all("table", {"class":"wikitable"})[1]
rows = table_classification_winners.find_all("tr")[1:]  # Skip header row

classification_data = []
for row in rows:
    cols = row.find_all("td")

    # This is for normal years with winners , between 1999 and 2005 inclusive because of Lance Armstrong
    if not(cols[0].find("a") is None or 1999 <= int(cols[0].text.strip()) <= 2005):
        year = cols[0].text.strip()
        country = cols[1].text.strip()
        cyclist = row.find("th").text.strip()
        
        # extracting the distance in km using regex
        m_dis = re.search(r'<td>([\d,.?]+)\s*km', str(cols[3]))
        # Convert distance to float to handle numerical operations
        distance_km = float(m_dis.group(1).replace(',', ''))
        
        # Extracting time or points
        td_time_or_points = str(cols[4])
        m_time  = re.search(r'(\d+)h\s*(\d+)\′\s*(\d+)″', td_time_or_points)
        m_points = re.search(r'>(\d+)<', td_time_or_points)


        # Determine if it's time or points and structure accordingly (save both formats)
        if m_time:
            h, m, s = map(int, m_time.groups())     # map helps to convert all to int (m_time.groups() returns the regex matched groups as strings)
            total_hours = h + m/60 + s/3600
            time_info = {
                "type": "time",
                "hours_total": round(total_hours, 2),
                "formatted": f"{h}h {m}′ {s}″"
            }
        elif m_points:
            time_info = {"type": "points", "points": int(m_points.group(1))}
        else:
            # Fallback in case neither time nor points are found
            time_info = {"type": "unknown", "raw": cols[4].text.strip()}

        classification_data.append({
            "year": year,
            "country": country,
            "cyclist": cyclist,
            "distance_km": distance_km,
            "time_info": time_info
        })

# export to pickle
with open('data/tdf_shortest_tour_clean.p',"wb") as f:
    pickle.dump(classification_data, f)


## `tdf_five_timers_clean.json`

Which of the Five-Time Tour de France Winners Had the Highest Average Winning Speed?

This question explores the elite group of riders—**Jacques Anquetil**, **Eddy Merckx**, **Bernard Hinault**, and **Miguel Induráin**—who each won the Tour de France five times.

For each legend, we will collect the distance and winning time for each of their victories from their corresponding Tour edition pages.

Each row in the dataset will contain:

- Cyclist’s name  
- Country  
- Year of victory  
- URL to the specific Tour edition page
- Winning time (in hours, standardized for comparability)  

In [34]:
import requests
from bs4 import BeautifulSoup
import re
import json

URL = "https://en.wikipedia.org/wiki/List_of_Tour_de_France_general_classification_winners#By_nationality"

headers = {"User-Agent": "Mozilla/5.0"}

page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, "html.parser")

div = soup.find("div", {"class": "mw-content-ltr mw-parser-output"})
table_multiple_winners = div.find_all("table", {"class": "wikitable"})[2]

# Select the first 4 rows after the header
rows = table_multiple_winners.find_all("tr")[1:5]

cyclist_data_five_timers = []

for row in rows:
    cyclist_name = re.search(r'value="([^"]+)">', str(row)).group(1)
    country = re.search(r'<abbr title="(.+)"', str(row)).group(1)

    victories_td = row.find_all("td")[-1]
    year_links = victories_td.find_all("a")

    cyclist_record = {
        "cyclist": cyclist_name,
        "country": country,
        "victories": []
    }

    for a in year_links:
        year_text = a.text.strip()
        full_url = f"https://en.wikipedia.org{a['href']}"

        year = re.search(r'/wiki/(\d{4})', a['href']).group(1)

        page_year = requests.get(full_url, headers=headers)
        soup_year = BeautifulSoup(page_year.content, "html.parser")

        content_year = soup_year.find("div", {"class": "mw-content-ltr mw-parser-output"})

        # Find the correct general classification table by caption
        tables = content_year.find_all("table", {"class": "wikitable"})
        table_general = None

        for t in tables:
            caption = t.find("caption")
            if caption and "Final general classification" in caption.get_text(): table_general = t ; break

        first_place = table_general.find_all("tr")[1]


        raw_time = first_place.find_all("td")[-1].text.strip()
        winning_time = raw_time.replace('"', '').replace('″', '').strip()
        

        cyclist_record["victories"].append({
            "year": year,
            "url": full_url,
            "winning_time": winning_time
        })

    cyclist_data_five_timers.append(cyclist_record)

with open("data/tdf_five_timers_clean.json", "w", encoding="utf-8") as f:
    json.dump(cyclist_data_five_timers, f, indent=3)

KeyboardInterrupt: 

## `tdf_winner_ages_clean.csv`

This dataset contains clean, structured data on the age at which each Tour de France general classification winner won the event, across all years of the race.


- Scrape the main “Tour de France general classification winners” table on Wikipedia to get:
  - The year of the Tour  
  - The name of the winning cyclist  
  - The link to each cyclist’s personal Wikipedia page  

- For each winner, follow the link to their personal Wikipedia page and:
  - Scrape the date of birth (from the “infobox vcard,” typically as `<span class="bday">YYYY-MM-DD</span>`)  
  - Subtract the year of birth from the Tour year to determine the cyclist’s age at victory *(ignore months for simplicity)*

All clean data is saved as:

`data/tdf_winner_ages_clean.csv`

This CSV file provides a ready dataset for exploring patterns and trends in the age of Tour de France champions across history.

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import csv

URL = "https://en.wikipedia.org/wiki/List_of_Tour_de_France_general_classification_winners#By_nationality"
headers = {"User-Agent": "Mozilla/5.0"}

page = requests.get(URL, headers=headers)
soup = BeautifulSoup(page.content, "html.parser")
div = soup.find("div", {"class":"mw-content-ltr mw-parser-output"})
table = div.find_all("table", {"class":"wikitable"})[1]
rows = table.find_all("tr")[1:]

output = []
for row in rows:
    cols = row.find_all("td")
    th = row.find("th")
    # Skip if there is no year link (no competition or stripped year)
    if not cols or cols[0].find("a") is None:
        continue
    
    year_str = cols[0].text.strip()
    year = int(year_str)

    if 1999 <= year <= 2005:
        continue
    country = cols[1].text.strip()
    cyclist_name = th.text.strip()
    cyclist_link = th.find("a")["href"]
    cyclist_url = f"https://en.wikipedia.org{cyclist_link}"

    # Get birthday from cyclist's personal Wiki page
    page_cyclist = requests.get(cyclist_url, headers=headers)
    soup_cyclist = BeautifulSoup(page_cyclist.content, "html.parser")
    bday_span = soup_cyclist.find("span", {"class":"bday"})
    if bday_span:
        bday = bday_span.text
        birth_year = int(bday.split('-')[0])
        age = year - birth_year
    else:
        bday = ''
        age = ''
    output.append([cyclist_name, country, year, bday, age])

# Export to CSV
header = ["Cyclist", "Country", "Year", "Birthdate", "Age"]
with open("data/tdf_winner_ages_clean.csv", "w", newline='', encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(output)